# PrepAI - Custom Model Training
This notebook automatically generates the required datasets and trains the 3 custom ML models for the PrepAI project.

In [ ]:
!pip install transformers datasets torch scikit-learn pandas accelerate sentence-transformers

### 1. Model 1: Answer Quality Classifier (DistilBERT)

In [ ]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

os.makedirs('data/answer_quality', exist_ok=True)

# Generate Data
data = [
    {'question': 'What is OOP?', 'answer': 'It is programming.', 'label': 0},
    {'question': 'What is OOP?', 'answer': 'Object oriented programming uses classes.', 'label': 1},
    {'question': 'What is OOP?', 'answer': 'OOP stands for Object-Oriented Programming and is based on concepts of objects.', 'label': 2},
    {'question': 'What is OOP?', 'answer': 'OOP is a programming paradigm based on objects and classes. It includes concepts like inheritance, polymorphism, encapsulation, and abstraction.', 'label': 3}
] * 50

df = pd.DataFrame(data)
df.to_csv('data/answer_quality/train.csv', index=False)

# Train Model
dataset = Dataset.from_pandas(df).train_test_split(test_size=0.2)
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

def tokenize_fn(examples):
    inputs = [f"Q: {q} A: {a}" for q, a in zip(examples['question'], examples['answer'])]
    return tokenizer(inputs, padding='max_length', truncation=True, max_length=128)

tokenized = dataset.map(tokenize_fn, batched=True)

training_args = TrainingArguments(
    output_dir='./saved_models/answer_quality_model',
    eval_strategy='epoch',
    num_train_epochs=3,
    per_device_train_batch_size=8
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test']
)
print('Training Answer Quality Model...')
trainer.train()
model.save_pretrained('./saved_models/answer_quality_model')
tokenizer.save_pretrained('./saved_models/answer_quality_model')
print('Model 1 Saved!')

### 2. Model 2: Interview Question Generator (GPT-2)

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset
import pandas as pd

os.makedirs('data/question_generator', exist_ok=True)

# Generate Data
text_data = """
[ROLE] Interviewer [TOPIC] Python [PREV_Q] None [ANSWER] None [QUALITY] None [NEXT_Q] What are the key features of Python?
[ROLE] Candidate [TOPIC] Python [PREV_Q] What are the key features of Python? [ANSWER] Python is interpreted, dynamically typed, and object-oriented. [QUALITY] Good [NEXT_Q] None
[ROLE] Interviewer [TOPIC] Python [PREV_Q] What are the key features of Python? [ANSWER] Python is interpreted, dynamically typed, and object-oriented. [QUALITY] Good [NEXT_Q] Can you explain dynamic typing?
""" * 100

model_name = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)

lines = [line for line in text_data.split('\n') if line.strip()]
df = pd.DataFrame({'text': lines})
dataset = Dataset.from_pandas(df)

def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir='./saved_models/question_generator_model',
    num_train_epochs=3,
    per_device_train_batch_size=4
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized
)
print('Training Question Generator Model...')
trainer.train()
model.save_pretrained('./saved_models/question_generator_model')
tokenizer.save_pretrained('./saved_models/question_generator_model')
print('Model 2 Saved!')

### 3. Model 3: Semantic Answer Relevance (Sentence-BERT)

In [ ]:
from sentence_transformers import SentenceTransformer, losses
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer
from datasets import Dataset
import pandas as pd

os.makedirs('data/answer_relevance', exist_ok=True)

# Generate Data
data = [
    {'question': 'Explain OOP.', 'answer': 'Object oriented programming.', 'label': 0.8},
    {'question': 'Explain OOP.', 'answer': 'I like eating pizza.', 'label': 0.0},
    {'question': 'What is Python?', 'answer': 'It is a programming language.', 'label': 0.9},
    {'question': 'What is Python?', 'answer': 'A type of snake.', 'label': 0.1}
] * 50

df = pd.DataFrame(data)
dataset = Dataset.from_pandas(df)

model = SentenceTransformer('all-MiniLM-L6-v2')
train_loss = losses.CosineSimilarityLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir='./saved_models/semantic_relevance_model',
    num_train_epochs=3,
    per_device_train_batch_size=16,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=dataset,
    loss=train_loss,
)
print('Training Semantic Relevance Model...')
trainer.train()
model.save_pretrained('./saved_models/semantic_relevance_model')
print('Model 3 Saved!')

### 4. Download Trained Models

In [ ]:
import shutil
from google.colab import files

print('Zipping models...')
shutil.make_archive('prepai_trained_models', 'zip', './saved_models')
files.download('prepai_trained_models.zip')
print('Download Started!')